- url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
- column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

In [63]:
# Regression - Predict a numerical value ,  continous nature
import pandas as pd
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'

column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

dataset = pd.read_csv(url, names=column_names, na_values='?', comment='\t',
                          sep=' ', skipinitialspace=True)

dataset.head()


,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


In [64]:
# data cleaning
dataset.isna().sum()



MPG             0
Cylinders       0
Displacement    0
Horsepower      6
Weight          0
Acceleration    0
Model Year      0
Origin          0
dtype: int64

In [65]:
dataset = dataset.dropna()

In [66]:
# split the dataset
train_dataset = dataset.sample(frac=0.8, random_state=0)
test_dataset = dataset.drop(train_dataset.index)

In [67]:
len(train_dataset), len(test_dataset)

(314, 78)

In [68]:
train_features = train_dataset.copy()
test_features = test_dataset.copy()
train_label = train_features.pop("MPG")
test_label = test_features.pop("MPG")

In [71]:
# Normalization ? 
import seaborn as sns


# sns.pairplot(train_dataset[['MPG', 'Cylinders', 'Displacement', 'Weight']], diag_kind="kde")

In [70]:
train_dataset

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Origin
146,28.0,4,90.0,75.0,2125.0,14.5,74,1
282,22.3,4,140.0,88.0,2890.0,17.3,79,1
69,12.0,8,350.0,160.0,4456.0,13.5,72,1
378,38.0,4,105.0,63.0,2125.0,14.7,82,1
331,33.8,4,97.0,67.0,2145.0,18.0,80,3
...,...,...,...,...,...,...,...,...
281,19.8,6,200.0,85.0,2990.0,18.2,79,1
229,16.0,8,400.0,180.0,4220.0,11.1,77,1
150,26.0,4,108.0,93.0,2391.0,15.5,74,3
145,32.0,4,83.0,61.0,2003.0,19.0,74,3


In [120]:
# Normalization  -1 , 1
import tensorflow as tf
normalize = tf.keras.layers.Normalization()

In [121]:
import numpy as np


normalize.adapt(np.array([[10,20,30,40,50]]))

In [122]:
b = np.array([[70]])

normalize(b)

<tf.Tensor: shape=(1, 5), dtype=float32, numpy=array([[6.e+08, 5.e+08, 4.e+08, 3.e+08, 2.e+08]], dtype=float32)>

In [123]:
normalize.mean.numpy()

array([[10., 20., 30., 40., 50.]], dtype=float32)

In [ ]:
text = "I have handling pets for a long time and i hold really cute pets like cat and dog, which are really cute"



In [143]:
import tensorflow as tf
import numpy as np

# 1. Sample Corpus
corpus = "we are learning word2vec with tensorflow word embeddings are useful".split()

# 2. Vocabulary
vocab = list(set(corpus))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
vocab_size = len(vocab)

# 3. Generate Skip-gram Pairs
def generate_pairs(corpus, window_size=2):
    pairs = []
    for i, word in enumerate(corpus):
        center = word_to_ix[word]
        context_range = range(max(0, i - window_size), min(len(corpus), i + window_size + 1))
        for j in context_range:
            if i != j:
                context = word_to_ix[corpus[j]]
                pairs.append((center, context))
    return pairs

pairs = generate_pairs(corpus)

# Convert to numpy arrays
X_train = np.array([x for x, _ in pairs])
y_train = np.array([y for _, y in pairs])

# 4. Build the Word2Vec Model in TensorFlow
class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2Vec, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=1)
        self.output_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs):
        x = self.embedding(inputs)
        x = tf.reshape(x, (-1, x.shape[-1]))  # Flatten
        return self.output_layer(x)

embedding_dim = 10
model = Word2Vec(vocab_size, embedding_dim)

# 5. Compile Model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# 6. Train the Model
model.fit(X_train, y_train, epochs=100, verbose=0)

# 7. Get Final Word Embeddings
embeddings = model.embedding.get_weights()[0]

# 8. Print Word Embeddings
for word, idx in word_to_ix.items():
    print(f"{word}: {embeddings[idx]}")


/Users/sachinmurali/anaconda3/envs/tensorflows/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


embeddings: [ 0.05653547 -0.12767285  0.04130336 -0.07489821  0.06716046 -0.00550153
  0.0219597  -0.00782783  0.08566479  0.00166854]
word: [-0.00012765 -0.06415778  0.02311473 -0.0595949  -0.06945553 -0.04460888
  0.05727309  0.06544093 -0.03073785  0.04639111]
we: [ 0.09489698  0.05675578 -0.01499265 -0.08682122  0.10143625 -0.02033995
 -0.01720472 -0.06781423  0.11839598  0.09712543]
are: [-0.08099899  0.11404866  0.08010967 -0.0358978   0.04105181  0.05674753
 -0.0443511  -0.00878059  0.02784064 -0.06772394]
learning: [-0.00775251 -0.02067519 -0.04592404  0.07791382  0.03638707 -0.10782056
 -0.08808769  0.00420669 -0.06299187 -0.0728863 ]
with: [ 0.06385412  0.00295325 -0.1523727   0.00793586  0.07880773  0.03440117
  0.06642389  0.02693126 -0.10339695  0.08994506]
useful: [-0.05247284  0.02111464  0.14248216 -0.05504094 -0.0691561  -0.10103177
  0.02311829  0.06643503  0.08784166  0.00994387]
tensorflow: [-0.0626353  -0.00277215  0.00105415  0.06381737 -0.09106991  0.03801638
  0

In [ ]:
### Transformer LLM (Large Lang Model)
# Lang Model  -  Organize, structure -  Generative Ai (generate text)
# Clay Model - organize, create , structure


# Model : Input ( numeric data)

#  LLM  --  text  -- numeric

# input   -->  hello, how are you  -- [1,2,3,4] -->

# [Encoder, Decoder]



# Architecture


#### #########################  ########
# Transleten lang ?   English to malayal
# 2001 -  Bang of Words
# Bag of words ---  I love this cat  ,  I love this dog
# vocab = [I , love , this, cat] --> sparse vector --> [1,1,1,1] , [1,1,1,0]
# advantnage : spam detect
# disavantage : semantics relatuionship , no context undertanding

# money in the bank ,   bank of of rivers








In [151]:
### Create Bag of words
text = "hello how are you doing today"
vocab = text.split()
vocab

['hello', 'how', 'are', 'you', 'doing', 'today']

In [152]:
# vocab = {word: i for i, word in enumerate(text.split())}
# vocab

In [189]:
new_text = "how boy you ".split()

[1 if word in vocab else 0 for word in new_text]





[1, 0, 1]